In [56]:
import pandas as pd
import csv
import joblib
import xgboost as xgb
from datetime import datetime
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
import seaborn as sns
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_fscore_support,
    matthews_corrcoef
)
import numpy as np
import tensorflow as tf
from tensorflow import keras
from imblearn.over_sampling import SMOTE
from collections import Counter
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from pyswarm import pso  # Librairie PSO



In [57]:
import pandas as pd

# Charger un fichier CSV
df = pd.read_csv('../datasets/Bias_correction_ucl.csv')

# Afficher les premières lignes du dataset


In [58]:
print(df.head())

   station        Date  Present_Tmax  Present_Tmin  LDAPS_RHmin  LDAPS_RHmax  \
0      1.0  2013-06-30          28.7          21.4    58.255688    91.116364   
1      2.0  2013-06-30          31.9          21.6    52.263397    90.604721   
2      3.0  2013-06-30          31.6          23.3    48.690479    83.973587   
3      4.0  2013-06-30          32.0          23.4    58.239788    96.483688   
4      5.0  2013-06-30          31.4          21.9    56.174095    90.155128   

   LDAPS_Tmax_lapse  LDAPS_Tmin_lapse  LDAPS_WS    LDAPS_LH  ...  LDAPS_PPT2  \
0         28.074101         23.006936  6.818887   69.451805  ...         0.0   
1         29.850689         24.035009  5.691890   51.937448  ...         0.0   
2         30.091292         24.565633  6.138224   20.573050  ...         0.0   
3         29.704629         23.326177  5.650050   65.727144  ...         0.0   
4         29.113934         23.486480  5.735004  107.965535  ...         0.0   

   LDAPS_PPT3  LDAPS_PPT4      lat    

In [43]:
df.replace("-", np.nan, inplace=True)
print("Nombre total de valeurs nulles :", df.isnull().sum().sum())

Nombre total de valeurs nulles : 1248


In [59]:
import pandas as pd
import numpy as np

# Remplacer les tirets par des valeurs nulles
df.replace("-", np.nan, inplace=True)

# Remplacer les valeurs nulles par la valeur la plus fréquente de chaque colonne
for col in df.columns:
    mode_val = df[col].mode()[0]  # Obtenir la valeur la plus fréquente (mode) pour la colonne
    df[col] = df[col].fillna(mode_val)  # Remplacer les valeurs nulles par la valeur la plus fréquente




In [61]:
print(df.dtypes)

station             float64
Date                 object
Present_Tmax        float64
Present_Tmin        float64
LDAPS_RHmin         float64
LDAPS_RHmax         float64
LDAPS_Tmax_lapse    float64
LDAPS_Tmin_lapse    float64
LDAPS_WS            float64
LDAPS_LH            float64
LDAPS_CC1           float64
LDAPS_CC2           float64
LDAPS_CC3           float64
LDAPS_CC4           float64
LDAPS_PPT1          float64
LDAPS_PPT2          float64
LDAPS_PPT3          float64
LDAPS_PPT4          float64
lat                 float64
lon                 float64
DEM                 float64
Slope               float64
Solar radiation     float64
Next_Tmax           float64
Next_Tmin           float64
dtype: object


In [62]:
# 1. Extraction de l'année, mois, jour, heure, minute, seconde de 'Timestamp'
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m-%d')
df['year'] = df['Date'].dt.year
df['month'] = df['Date'].dt.month
df['day'] = df['Date'].dt.day

In [63]:
df = df.drop(columns=df.select_dtypes(include=['datetime']).columns)

In [64]:
df.to_csv('df_cleaned.csv', index=False)

In [65]:
print(df.dtypes)

station             float64
Present_Tmax        float64
Present_Tmin        float64
LDAPS_RHmin         float64
LDAPS_RHmax         float64
LDAPS_Tmax_lapse    float64
LDAPS_Tmin_lapse    float64
LDAPS_WS            float64
LDAPS_LH            float64
LDAPS_CC1           float64
LDAPS_CC2           float64
LDAPS_CC3           float64
LDAPS_CC4           float64
LDAPS_PPT1          float64
LDAPS_PPT2          float64
LDAPS_PPT3          float64
LDAPS_PPT4          float64
lat                 float64
lon                 float64
DEM                 float64
Slope               float64
Solar radiation     float64
Next_Tmax           float64
Next_Tmin           float64
year                  int32
month                 int32
day                   int32
dtype: object


In [66]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Définir les caractéristiques (X) et les cibles (y)
X = df.drop(columns=["Next_Tmin", "Next_Tmax"])  # Exemple de caractéristiques d'entrée
y = df[['Next_Tmin', 'Next_Tmax']]  # Variables cibles : température minimale et maximale futures

# Séparer les données en ensembles d'entraînement et de test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Appliquer le MinMaxScaler
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [67]:
# Créer et entraîner le modèle de régression linéaire
model = LinearRegression()
model.fit(X_train, y_train)
# Faire des prédictions
y_pred = model.predict(X_test)



In [68]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# MAE : Erreur absolue moyenne
mae = mean_absolute_error(y_test, y_pred)

# MSE : Erreur quadratique moyenne
mse = mean_squared_error(y_test, y_pred)

# RMSE : Racine de l’erreur quadratique moyenne
rmse = np.sqrt(mse)

# R² : Coefficient de détermination
r2 = r2_score(y_test, y_pred)

# Affichage
print("📏 MAE  :", mae)
print("📏 MSE  :", mse)
print("📏 RMSE :", rmse)
print("📈 R²   :", r2)


📏 MAE  : 1.019202658173496
📏 MSE  : 1.9428435544526645
📏 RMSE : 1.3938592305009372
📈 R²   : 0.7652846556764124


In [70]:
# Créer et entraîner le modèle de régression linéaire
model = LinearRegression()
model.fit(X_train_scaled, y_train)
# Faire des prédictions
y_pred = model.predict(X_test_scaled)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# MAE : Erreur absolue moyenne
mae = mean_absolute_error(y_test, y_pred)

# MSE : Erreur quadratique moyenne
mse = mean_squared_error(y_test, y_pred)

# RMSE : Racine de l’erreur quadratique moyenne
rmse = np.sqrt(mse)

# R² : Coefficient de détermination
r2 = r2_score(y_test, y_pred)

# Affichage
print("📏 MAE  :", mae)
print("📏 MSE  :", mse)
print("📏 RMSE :", rmse)
print("📈 R²   :", r2)



📏 MAE  : 1.019202658173497
📏 MSE  : 1.9428435544526517
📏 RMSE : 1.3938592305009325
📈 R²   : 0.765284655676414


In [71]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor

# Création du modèle multi-sortie
rf = RandomForestRegressor(n_estimators=100, random_state=42)
multi_rf = MultiOutputRegressor(rf)

# Entraînement
multi_rf.fit(X_train, y_train)

# Prédictions
y_pred = multi_rf.predict(X_test)

# Évaluation
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

print("📏 MAE :", mae)
print("📏 MSE :", mse)
print("📏 RMSE :", rmse)
print("📈 R²  :", r2)


📏 MAE : 0.6264945196647322
📏 MSE : 0.6961436299161828
📏 RMSE : 0.8343522217362298
📈 R²  : 0.9124237583763201


In [72]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor

# Création du modèle multi-sortie
rf = RandomForestRegressor(n_estimators=100, random_state=42)
multi_rf = MultiOutputRegressor(rf)

# Entraînement
multi_rf.fit(X_train_scaled, y_train)

# Prédictions
y_pred = multi_rf.predict(X_test_scaled)

# Évaluation
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

print("📏 MAE :", mae)
print("📏 MSE :", mse)
print("📏 RMSE :", rmse)
print("📈 R²  :", r2)


📏 MAE : 0.6266018697614439
📏 MSE : 0.6960525348162475
📏 RMSE : 0.8342976296359996
📈 R²  : 0.9124048204215205


In [73]:
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Création du modèle XGBoost pour multi-sortie
xgb = XGBRegressor(objective='reg:squarederror', n_estimators=100, learning_rate=0.1, random_state=42)
multi_xgb = MultiOutputRegressor(xgb)

# Entraînement
multi_xgb.fit(X_train, y_train)

# Prédictions
y_pred = multi_xgb.predict(X_test)

# Évaluation
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

# Résultats
print("📏 MAE :", mae)
print("📏 MSE :", mse)
print("📏 RMSE :", rmse)
print("📈 R²  :", r2)


📏 MAE : 0.5612986087799072
📏 MSE : 0.5706807374954224
📏 RMSE : 0.7554341384233455
📈 R²  : 0.9297348260879517


In [74]:
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Création du modèle XGBoost pour multi-sortie
xgb = XGBRegressor(objective='reg:squarederror', n_estimators=100, learning_rate=0.1, random_state=42)
multi_xgb = MultiOutputRegressor(xgb)

# Entraînement
multi_xgb.fit(X_train_scaled, y_train)

# Prédictions
y_pred = multi_xgb.predict(X_test_scaled)

# Évaluation
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

# Résultats
print("📏 MAE :", mae)
print("📏 MSE :", mse)
print("📏 RMSE :", rmse)
print("📈 R²  :", r2)


📏 MAE : 0.5612986087799072
📏 MSE : 0.5706807374954224
📏 RMSE : 0.7554341384233455
📈 R²  : 0.9297348260879517


In [75]:
from sklearn.neural_network import MLPRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Création du modèle MLP
mlp = MLPRegressor(hidden_layer_sizes=(100, 100), max_iter=1000, random_state=42)
multi_mlp = MultiOutputRegressor(mlp)

# Entraînement
multi_mlp.fit(X_train, y_train)

# Prédictions
y_pred = multi_mlp.predict(X_test)

# Évaluation
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

# Résultats
print("📏 MAE :", mae)
print("📏 MSE :", mse)
print("📏 RMSE :", rmse)
print("📈 R²  :", r2)



📏 MAE : 3.4540781904384557
📏 MSE : 15.954651506055452
📏 RMSE : 3.9943274159807496
📈 R²  : -1.2965822842531416


In [76]:
from sklearn.neural_network import MLPRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Création du modèle MLP
mlp = MLPRegressor(hidden_layer_sizes=(100, 100), max_iter=1000, random_state=42)
multi_mlp = MultiOutputRegressor(mlp)

# Entraînement
multi_mlp.fit(X_train_scaled, y_train)

# Prédictions
y_pred = multi_mlp.predict(X_test_scaled)

# Évaluation
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

# Résultats
print("📏 MAE :", mae)
print("📏 MSE :", mse)
print("📏 RMSE :", rmse)
print("📈 R²  :", r2)



📏 MAE : 0.7904854881535496
📏 MSE : 1.0683361617932112
📏 RMSE : 1.033603483833724
📈 R²  : 0.8673234999915118


Optimisation

In [33]:
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Création du modèle XGBoost pour multi-sortie
xgb = XGBRegressor(max_depth=20, max_features='sqrt', n_estimators=1000,random_state=42)
multi_xgb = MultiOutputRegressor(xgb)

# Entraînement
multi_xgb.fit(X_train, y_train)

# Prédictions
y_pred = multi_xgb.predict(X_test)

# Évaluation
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

# Résultats
print("📏 MAE :", mae)
print("📏 MSE :", mse)
print("📏 RMSE :", rmse)
print("📈 R²  :", r2)


c:\Users\T U F\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:183: UserWarning: [18:22:24] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "max_features" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\T U F\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:183: UserWarning: [18:22:29] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "max_features" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


📏 MAE : 0.6777868866920471
📏 MSE : 0.8933935165405273
📏 RMSE : 0.94519496218533
📈 R²  : 0.8900277614593506


In [35]:
# Pourcentage d’importance des features pour Next_Tmax
first_xgb_model = multi_xgb.estimators_[0]
importances_1 = first_xgb_model.feature_importances_
importances_1 = importances_1 / importances_1.sum() * 100  # normaliser en %

print("🎯 Importances des features pour la prédiction de Next_Tmax :")
for name, score in sorted(zip(X_train.columns, importances_1), key=lambda x: x[1], reverse=True):
    print(f"{name}: {score:.2f} %")

# Pourcentage d’importance des features pour Next_Tmin
second_xgb_model = multi_xgb.estimators_[1]
importances_2 = second_xgb_model.feature_importances_
importances_2 = importances_2 / importances_2.sum() * 100  # normaliser en %

print("\n🎯 Importances des features pour la prédiction de Next_Tmin :")
for name, score in sorted(zip(X_train.columns, importances_2), key=lambda x: x[1], reverse=True):
    print(f"{name}: {score:.2f} %")


🎯 Importances des features pour la prédiction de Next_Tmax :
LDAPS_Tmin_lapse: 67.26 %
Present_Tmin: 6.58 %
year: 2.15 %
DEM: 1.97 %
Slope: 1.79 %
LDAPS_PPT2: 1.73 %
LDAPS_PPT1: 1.61 %
LDAPS_PPT4: 1.58 %
lon: 1.47 %
LDAPS_CC4: 1.32 %
lat: 1.23 %
month: 1.20 %
day: 1.17 %
Solar radiation: 1.17 %
LDAPS_CC1: 1.12 %
LDAPS_CC2: 0.92 %
LDAPS_WS: 0.86 %
LDAPS_PPT3: 0.85 %
LDAPS_CC3: 0.84 %
LDAPS_RHmin: 0.83 %
Present_Tmax: 0.57 %
LDAPS_Tmax_lapse: 0.50 %
station: 0.47 %
LDAPS_LH: 0.40 %
LDAPS_RHmax: 0.40 %

🎯 Importances des features pour la prédiction de Next_Tmin :
LDAPS_Tmax_lapse: 53.33 %
year: 5.88 %
day: 5.83 %
LDAPS_RHmax: 5.09 %
LDAPS_PPT2: 2.93 %
LDAPS_CC1: 2.78 %
LDAPS_CC3: 2.77 %
month: 2.04 %
Present_Tmax: 1.95 %
LDAPS_PPT3: 1.82 %
LDAPS_PPT1: 1.78 %
LDAPS_CC4: 1.65 %
Solar radiation: 1.52 %
LDAPS_WS: 1.46 %
LDAPS_CC2: 1.33 %
LDAPS_Tmin_lapse: 1.12 %
LDAPS_PPT4: 1.09 %
LDAPS_LH: 0.93 %
Present_Tmin: 0.86 %
LDAPS_RHmin: 0.78 %
Slope: 0.71 %
lon: 0.69 %
lat: 0.62 %
DEM: 0.54 %
stati